In [1]:
import numpy as np
from matplotlib import pyplot as plt
import math
import scipy.constants as constants
import numba
get_ipython().run_line_magic('matplotlib', 'auto')

Using matplotlib backend: MacOSX


I don't even know whether this is a 1D or 2D problem. As everything is propagating only in the z direction, I guess this is a 1D problem, but 2D vectors. I will call this 1D for now. Because of that, it should be a 1D plot, hence vvv

In [18]:
def init(xmax):
    plt.xlim((0, xmax-1))
    plt.grid('on')
    ax.set_xlabel('Grid Cells ($z$)')
    plt.show()

## Pulse (trivial, ABC, TFSF)

In [ ]:
# stability requirements
dx = 20e-9 # ~ nm grid size
dt_si = dx*0.5/constants.c # ~3.3e-17 s in one grid

# pulse 
dt_fs = dt_si/constants.femto # ! ~0.033 fs in one grid
spread = 2/dt_fs # 1/df_fs = num of grids in 1 fs, spread = num of grids in 2 fs ~ 60 grids
t0 = spread*6 # offset by 60*6=360 grids
freq_in = 2*math.pi* 200* constants.tera # angular frequency = 1.256e15 rad s^-1
w_scale = freq_in*dt_si # 0.042 rad per grid

# set time in grid unit
nsteps = 2000
t = np.arange(0,nsteps+1)

@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*w_scale)
    return source

#plt.xlim(0,20)
plt.clf()
plt.plot(t, get_source(t))
plt.show()
print(t0, spread)

In [ ]:
#kappa = 1
@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Bx, By, axion):
    Ex_phy = (Ex + axion*Bx)/(1+axion**2)
    Ey_phy = (Ey + axion*By)/(1+axion**2)
    Bx_phy = (Bx - axion*Ex)/(1+axion**2)
    By_phy = (By - axion*Ey)/(1+axion**2)
    
    return Ex_phy, Ey_phy, Bx_phy, By_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion):
    Ex = Ex_phy - axion*Bx_phy
    Ey = Ey_phy - axion*By_phy
    Bx = Bx_phy + axion*Ex_phy
    By = By_phy + axion*Ey_phy
    
    return Ex, Ey, Bx, By

In [ ]:
@numba.jit(nopython=True)
def Eupdate1d(Ex, Ey, Bx, By):
    for k in range(1,kmax-1):
        Ex[k] = Ex[k] + 0.5 * (By[k-1] - By[k])
        Ey[k] = Ey[k] + 0.5 * (Bx[k] - Bx[k-1])
    return Ex, Ey
@numba.jit(nopython=True)
def Bupdate1d(Ex, Ey, Bx, By):
    for k in range(0,kmax-1):
        Bx[k] = Bx[k] + 0.5 * (Ey[k+1] - Ey[k])
        By[k] = By[k] + 0.5 * (Ex[k] - Ex[k+1])
    return Bx, By
@numba.jit(nopython=True)
def Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy):
    for k in range(1,kmax-1):
        axion[k] = 2*axion[k] - axion_past[k] + 0.25*(axion[k+1] - 2*axion[k] + axion[k-1])\
        - dt**2*(Ex_phy[k]*Bx_phy[k] + Ey_phy[k]*By_phy[k]) - dt**2*(m**2*axion[k])
        
    return axion
    

In [ ]:
kmax = 800
nsteps = 2000
Ex = Ey = Ex_phy = Ey_phy = np.zeros(kmax, float)
Bx = By = Bx_phy = By_phy = np.zeros(kmax, float)
axion = np.zeros(kmax, float)
xrange = np.linspace(0,kmax, kmax)

# axion
m = 1
kappa = 1

# source
sourceidx = int(kmax/4)

plt.clf()
plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex_phy,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
init(kmax)

for t in range(0,nsteps+1):
    source = get_source(t)
    source2 = get_source(t+0.5)
    
    if t == 0:
        axion_past = np.zeros(kmax, float)
    # ABC
    Exleft_0 = Ex[1]
    Eyleft_0 = Ey[1]
    Exright_0 = Ex[-2]
    Eyright_0 = Ey[-2]
        
        
    # update E
    Ex, Ey = Eupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    # inject E source
    Ex_phy[sourceidx] = Ex_phy[sourceidx] - 0.5*source2
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # ABC
    if t == 0:
        Exleft_ = None
        Eyleft_ = None
        Exright_ = None
        Eyright_ = None
    if t != 0:
        Ex[0] = Exleft_
        Ey[0] = Eyleft_
        Ex[-1] = Exright_
        Ey[-1] = Eyright_
    
    Exleft_ = Exleft_0
    Eyleft_ = Eyleft_0
    Exright_ = Exright_0
    Eyright_ = Eyright_0
    
    # update B
    Bx, By = Bupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # TFSF
    By_phy[sourceidx-1] = By_phy[sourceidx-1] - 0.5*source
    Bx_phy[sourceidx-1] = Bx_phy[sourceidx-1] - 0.5*source
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # update A
    axion = Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy)
    
    if t == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    # plot
    if t % cycle == 0:
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        im.set_ydata(Ex_phy)
        im2.set_ydata(axion)
        ax.set_title("frame time {}".format(t))
        plt.show()
        plt.pause(0.05)
print('done')

## Pulse ($B_{external}$, ABC, TFSF)

In [4]:
# stability requirements
dx = 20e-9 # ~ nm grid size
dt_si = dx*0.5/constants.c # ~3.3e-17 s in one grid

# pulse 
dt_fs = dt_si/constants.femto # ! ~0.033 fs in one grid
spread = 2/dt_fs # 1/df_fs = num of grids in 1 fs, spread = num of grids in 2 fs ~ 60 grids
t0 = spread*6 # offset by 60*6=360 grids
freq_in = 2*math.pi* 200* constants.tera # angular frequency = 1.256e15 rad s^-1
w_scale = freq_in*dt_si # 0.042 rad per grid

# set time in grid unit
nsteps = 2000
t = np.arange(0,nsteps+1)

@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*w_scale)
    return source

#plt.xlim(0,20)
plt.clf()
plt.plot(t, get_source(t))
plt.show()
print(t0, spread)

359.75094960000007 59.95849160000001


In [5]:
#kappa = 1
@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Bx, By, axion):
    Ex_phy = (Ex + axion*Bx)/(1+axion**2)
    Ey_phy = (Ey + axion*By)/(1+axion**2)
    Bx_phy = (Bx - axion*Ex)/(1+axion**2)
    By_phy = (By - axion*Ey)/(1+axion**2)
    
    return Ex_phy, Ey_phy, Bx_phy, By_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion):
    Ex = Ex_phy - axion*Bx_phy
    Ey = Ey_phy - axion*By_phy
    Bx = Bx_phy + axion*Ex_phy
    By = By_phy + axion*Ey_phy
    
    return Ex, Ey, Bx, By

@numba.jit(nopython=True)
def Eupdate1d(Ex, Ey, Bx, By):
    for k in range(1,kmax-1):
        Ex[k] = Ex[k] + 0.5 * (By[k-1] - By[k])
        Ey[k] = Ey[k] + 0.5 * (Bx[k] - Bx[k-1])
    return Ex, Ey
@numba.jit(nopython=True)
def Bupdate1d(Ex, Ey, Bx, By):
    for k in range(0,kmax-1):
        Bx[k] = Bx[k] + 0.5 * (Ey[k+1] - Ey[k])
        By[k] = By[k] + 0.5 * (Ex[k] - Ex[k+1])
    return Bx, By
@numba.jit(nopython=True)
def Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy):
    for k in range(1,kmax-1):
        axion[k] = 2*axion[k] - axion_past[k] + 0.25*(axion[k+1] - 2*axion[k] + axion[k-1])\
        - dt**2*(Ex_phy[k]*Bx_phy[k] + Ey_phy[k]*By_phy[k]) - dt**2*(m**2*axion[k])
        
    return axion
    

In [7]:
nsteps=4000
kmax = 2000
Ex_phy = Ey_phy = np.zeros(kmax, float)
By_phy = np.zeros(kmax, float)
Bx_phy = np.zeros(kmax, float) # Bext source
axion = np.zeros(kmax, float)

Ex= Ey= Bx= By = np.zeros(kmax, float)

xrange = np.linspace(0,kmax, kmax)

# axion
m = 1 # eV
kappa = 1
hbar = 6.582e-16 # eVs
dt = dt_si/hbar # ~0.05 eV^-1

# source
sourceidx = int(kmax/4)

plt.clf()
plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex_phy,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
im.set_color('blue')
im2.set_color('orange')
plt.legend(['$E_{phy,x}$','axion'])
plt.ylim(-1,1)
init(kmax)

for t in range(0,nsteps+1):
    source = get_source(t)
    source2 = get_source(t+0.5)
    
    if t == 0:
        axion_past = np.zeros(kmax, float)
    # ABC
    Exleft_0 = Ex[1]
    Eyleft_0 = Ey[1]
    Exright_0 = Ex[-2]
    Eyright_0 = Ey[-2]
        
        
    # update E
    Ex, Ey = Eupdate1d(Ex, Ey, Bx, By)
    # convert to physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    # inject E source
    Ex_phy[sourceidx] = Ex_phy[sourceidx] - 0.5*source2
    
    #convert back to hat's fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # ABC
    if t == 0:
        Exleft_ = None
        Eyleft_ = None
        Exright_ = None
        Eyright_ = None
    if t != 0:
        Ex[0] = Exleft_
        Ey[0] = Eyleft_
        Ex[-1] = Exright_
        Ey[-1] = Eyright_
    
    Exleft_ = Exleft_0
    Eyleft_ = Eyleft_0
    Exright_ = Exright_0
    Eyright_ = Eyright_0
    
    # update B
    Bx, By = Bupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # TFSF
    By_phy[sourceidx-1] = By_phy[sourceidx-1] - 0.5*source
    #Bx_phy[sourceidx-1] = Bx_phy[sourceidx-1] - 0.5*source
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    
    if t == 1000:
        # convert to physical fields
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        # inject B source
        Bx_phy = Bx_phy + 0.5*np.ones_like(Bx_phy)
        #convert back to hat's fields
        Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # update A
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    axion = Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy)
    
    if t == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    # plot
    if t % cycle == 0:
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        im.set_ydata(Ex_phy) # blue
        im2.set_ydata(axion) # orange
        ax.set_title("frame time {}".format(t))
        plt.savefig('/Users/szechingaudreyfung/Desktop/PHYS 879 HPC/Projects/plots/1dpulse/1dpulse{}.png'.format(t))
        plt.show()
        plt.pause(0.05)
print('done')

done


Comments: after setting the pulse in SI units, $\Delta t$ is now extremely short, ~ $3.3 \times 10^{-17}s$, as axion conversion depends on $\Delta t^2$, there's basically no conversion, which is uninteresting, albeit realistic, that is a reason why we want microwave long source for conversion, with m ~ 1eV. Later, I will use a source that has a comparable wavelength with a 1eV axion conversion in the plane wave case.

Comments2: actually, as my problem is in natural units, time should also be in natural units. OK, turns out $\Delta t$ is around $0.05 eV^{-1}$, so there is measurable conversion!

## Soft source (trivial, ABC, TFSF)

soft source is not very insightful, so I will skip the SI conversion for now.

In [ ]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

@numba.jit(nopython=True)
def get_source(t):
    t0 = nsteps/5
    spread = nsteps/30
    source = np.exp(-0.5*(t-t0)**2/spread**2)
    return source

plt.clf()
plt.plot(np.arange(nsteps), get_source(np.arange(nsteps)))
plt.xlim(0,nsteps)
plt.ylim(-1,1)
plt.show()

In [ ]:
#kappa = 1
@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Bx, By, axion):
    Ex_phy = (Ex + axion*Bx)/(1+axion**2)
    Ey_phy = (Ey + axion*By)/(1+axion**2)
    Bx_phy = (Bx - axion*Ex)/(1+axion**2)
    By_phy = (By - axion*Ey)/(1+axion**2)
    
    return Ex_phy, Ey_phy, Bx_phy, By_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion):
    Ex = Ex_phy - axion*Bx_phy
    Ey = Ey_phy - axion*By_phy
    Bx = Bx_phy + axion*Ex_phy
    By = By_phy + axion*Ey_phy
    
    return Ex, Ey, Bx, By

@numba.jit(nopython=True)
def Eupdate1d(Ex, Ey, Bx, By):
    for k in range(1,kmax-1):
        Ex[k] = Ex[k] + 0.5 * (By[k-1] - By[k])
        Ey[k] = Ey[k] + 0.5 * (Bx[k] - Bx[k-1])
    return Ex, Ey
@numba.jit(nopython=True)
def Bupdate1d(Ex, Ey, Bx, By):
    for k in range(0,kmax-1):
        Bx[k] = Bx[k] + 0.5 * (Ey[k+1] - Ey[k])
        By[k] = By[k] + 0.5 * (Ex[k] - Ex[k+1])
    return Bx, By
@numba.jit(nopython=True)
def Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy):
    for k in range(1,kmax-1):
        axion[k] = 2*axion[k] - axion_past[k] + 0.25*(axion[k+1] - 2*axion[k] + axion[k-1])\
        - dt**2*(Ex_phy[k]*Bx_phy[k] + Ey_phy[k]*By_phy[k]) - dt**2*(m**2*axion[k])
        
    return axion
    

In [ ]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6


kmax = 800
nsteps = 2000
Ex = Ey = Ex_phy = Ey_phy = np.zeros(kmax, float)
Bx = By = Bx_phy = By_phy = np.zeros(kmax, float)
axion = np.zeros(kmax, float)
xrange = np.linspace(0,kmax, kmax)

# stability
dx = 1
dt = 0.5*dx

# axion
m = 1
kappa = 1

# source
sourceidx = int(kmax/4)

plt.clf()
plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex_phy,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
init(kmax)

for t in range(0,nsteps+1):
    source = get_source(t)
    source2 = get_source(t+0.5)
    
    if t == 0:
        axion_past = np.zeros(kmax, float)
    # ABC
    Exleft_0 = Ex[1]
    Eyleft_0 = Ey[1]
    Exright_0 = Ex[-2]
    Eyright_0 = Ey[-2]
        
        
    # update E
    Ex, Ey = Eupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    # inject E source
    Ex_phy[sourceidx] = Ex_phy[sourceidx] + 0.5*source2
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # ABC
    if t == 0:
        Exleft_ = None
        Eyleft_ = None
        Exright_ = None
        Eyright_ = None
    if t != 0:
        Ex[0] = Exleft_
        Ey[0] = Eyleft_
        Ex[-1] = Exright_
        Ey[-1] = Eyright_
    
    Exleft_ = Exleft_0
    Eyleft_ = Eyleft_0
    Exright_ = Exright_0
    Eyright_ = Eyright_0
    
    # update B
    Bx, By = Bupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # TFSF
    By_phy[sourceidx-1] = By_phy[sourceidx-1] + 0.5*source
    Bx_phy[sourceidx-1] = Bx_phy[sourceidx-1] + 0.5*source
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # update A
    axion = Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy)
    
    if t == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    # plot
    if t % cycle == 0:
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        im.set_ydata(Ex_phy)
        im2.set_ydata(axion)
        ax.set_title("frame time {}".format(t))
        plt.show()
        plt.pause(0.05)
print('done')

## Soft source ($B_{external}$, ABC, TFSF)

In [ ]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

@numba.jit(nopython=True)
def get_source(t):
    t0 = nsteps/5
    spread = nsteps/30
    source = np.exp(-0.5*(t-t0)**2/spread**2)
    return source

plt.clf()
plt.plot(np.arange(nsteps), get_source(np.arange(nsteps)))
plt.xlim(0,nsteps)
plt.ylim(-1,1)
plt.show()

In [ ]:
#kappa = 1
@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Bx, By, axion):
    Ex_phy = (Ex + axion*Bx)/(1+axion**2)
    Ey_phy = (Ey + axion*By)/(1+axion**2)
    Bx_phy = (Bx - axion*Ex)/(1+axion**2)
    By_phy = (By - axion*Ey)/(1+axion**2)
    
    return Ex_phy, Ey_phy, Bx_phy, By_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion):
    Ex = Ex_phy - axion*Bx_phy
    Ey = Ey_phy - axion*By_phy
    Bx = Bx_phy + axion*Ex_phy
    By = By_phy + axion*Ey_phy
    
    return Ex, Ey, Bx, By

@numba.jit(nopython=True)
def Eupdate1d(Ex, Ey, Bx, By):
    for k in range(1,kmax-1):
        Ex[k] = Ex[k] + 0.5 * (By[k-1] - By[k])
        Ey[k] = Ey[k] + 0.5 * (Bx[k] - Bx[k-1])
    return Ex, Ey
@numba.jit(nopython=True)
def Bupdate1d(Ex, Ey, Bx, By):
    for k in range(0,kmax-1):
        Bx[k] = Bx[k] + 0.5 * (Ey[k+1] - Ey[k])
        By[k] = By[k] + 0.5 * (Ex[k] - Ex[k+1])
    return Bx, By
@numba.jit(nopython=True)
def Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy):
    for k in range(1,kmax-1):
        axion[k] = 2*axion[k] - axion_past[k] + 0.25*(axion[k+1] - 2*axion[k] + axion[k-1])\
        - dt**2*(Ex_phy[k]*Bx_phy[k] + Ey_phy[k]*By_phy[k]) - dt**2*(m**2*axion[k])
        
    return axion
    

In [ ]:
nsteps = 4000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6


kmax = 2000
Ex_phy = Ey_phy = np.zeros(kmax, float)
By_phy = np.zeros(kmax, float)
Bx_phy = np.zeros(kmax, float) # Bext source
axion = np.zeros(kmax, float)

Ex= Ey= Bx= By = np.zeros(kmax, float)

xrange = np.linspace(0,kmax, kmax)

# stability
dx = 1
dt = 0.5*dx

# axion
m = 1
kappa = 1

# source
sourceidx = int(kmax/4)

plt.clf()
plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex_phy,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
im.set_color('blue')
im2.set_color('orange')
init(kmax)

for t in range(0,nsteps+1):
    source = get_source(t)
    source2 = get_source(t+0.5)
    
    if t == 0:
        axion_past = np.zeros(kmax, float)
    # ABC
    Exleft_0 = Ex[1]
    Eyleft_0 = Ey[1]
    Exright_0 = Ex[-2]
    Eyright_0 = Ey[-2]
        
        
    # update E
    Ex, Ey = Eupdate1d(Ex, Ey, Bx, By)
    # convert to physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    # inject E source
    Ex_phy[sourceidx] = Ex_phy[sourceidx] + 0.5*source2
    
    #convert back to hat's fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # ABC
    if t == 0:
        Exleft_ = None
        Eyleft_ = None
        Exright_ = None
        Eyright_ = None
    if t != 0:
        Ex[0] = Exleft_
        Ey[0] = Eyleft_
        Ex[-1] = Exright_
        Ey[-1] = Eyright_
    
    Exleft_ = Exleft_0
    Eyleft_ = Eyleft_0
    Exright_ = Exright_0
    Eyright_ = Eyright_0
    
    # update B
    Bx, By = Bupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # TFSF
    By_phy[sourceidx-1] = By_phy[sourceidx-1] + 0.5*source
    #Bx_phy[sourceidx-1] = Bx_phy[sourceidx-1] + 0.5*source
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    if t == 1000:
        # convert to physical fields
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        # inject B source
        Bx_phy = Bx_phy + 0.5*np.ones_like(Bx_phy)
        #convert back to hat's fields
        Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
        
    # update A
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    axion = Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy)
    
    if t == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    # plot
    if t % cycle == 0:
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        im.set_ydata(Ex_phy) # blue
        im2.set_ydata(axion) # orange
        ax.set_title("frame time {}".format(t))
        plt.show()
        plt.pause(0.05)
print('done')

## Plane wave (trivial, ABC, TFSF)

Should probably try resonant conversion

In [ ]:
def get_source(t):
    E0 = 0.5
    wavelength=800
    t0 = 0
    #w = 2*np.pi/(wavelength)
    w = 0.01*np.pi
    #mask = (t>t0)*1
    source = E0*np.sin(w*t)#*mask
    return source

plt.close()
spread = 60
t0 = spread*6
nsteps=2000
plt.plot(np.arange(0,nsteps), get_source(np.arange(0,nsteps)), label='source')
plt.ylim(-1,1)
plt.legend()
plt.show()

In [ ]:
#kappa = 1
@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Bx, By, axion):
    Ex_phy = (Ex + axion*Bx)/(1+axion**2)
    Ey_phy = (Ey + axion*By)/(1+axion**2)
    Bx_phy = (Bx - axion*Ex)/(1+axion**2)
    By_phy = (By - axion*Ey)/(1+axion**2)
    
    return Ex_phy, Ey_phy, Bx_phy, By_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion):
    Ex = Ex_phy - axion*Bx_phy
    Ey = Ey_phy - axion*By_phy
    Bx = Bx_phy + axion*Ex_phy
    By = By_phy + axion*Ey_phy
    
    return Ex, Ey, Bx, By

@numba.jit(nopython=True)
def Eupdate1d(Ex, Ey, Bx, By):
    for k in range(1,kmax-1):
        Ex[k] = Ex[k] + 0.5 * (By[k-1] - By[k])
        Ey[k] = Ey[k] + 0.5 * (Bx[k] - Bx[k-1])
    return Ex, Ey
@numba.jit(nopython=True)
def Bupdate1d(Ex, Ey, Bx, By):
    for k in range(0,kmax-1):
        Bx[k] = Bx[k] + 0.5 * (Ey[k+1] - Ey[k])
        By[k] = By[k] + 0.5 * (Ex[k] - Ex[k+1])
    return Bx, By
@numba.jit(nopython=True)
def Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy):
    for k in range(1,kmax-1):
        axion[k] = 2*axion[k] - axion_past[k] + 0.25*(axion[k+1] - 2*axion[k] + axion[k-1])\
        - dt**2*(Ex_phy[k]*Bx_phy[k] + Ey_phy[k]*By_phy[k]) - dt**2*(m**2*axion[k])
        
    return axion
    

In [ ]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6


kmax = 800
nsteps = 2000
Ex = Ey = Ex_phy = Ey_phy = np.zeros(kmax, float)
Bx = By = Bx_phy = By_phy = np.zeros(kmax, float)
axion = np.zeros(kmax, float)
xrange = np.linspace(0,kmax, kmax)

# stability
dx = 1
dt = 0.5*dx

# axion
m = 1
kappa = 1

# source
sourceidx = int(kmax/4)

plt.clf()
plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex_phy,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
init(kmax)

for t in range(0,nsteps+1):
    source = get_source(t)
    source2 = get_source(t+0.5)
    
    if t == 0:
        axion_past = np.zeros(kmax, float)
    # ABC
    Exleft_0 = Ex[1]
    Eyleft_0 = Ey[1]
    Exright_0 = Ex[-2]
    Eyright_0 = Ey[-2]
        
        
    # update E
    Ex, Ey = Eupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    # inject E source
    Ex_phy[sourceidx] = Ex_phy[sourceidx] + 0.5*source2
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # ABC
    if t == 0:
        Exleft_ = None
        Eyleft_ = None
        Exright_ = None
        Eyright_ = None
    if t != 0:
        Ex[0] = Exleft_
        Ey[0] = Eyleft_
        Ex[-1] = Exright_
        Ey[-1] = Eyright_
    
    Exleft_ = Exleft_0
    Eyleft_ = Eyleft_0
    Exright_ = Exright_0
    Eyright_ = Eyright_0
    
    # update B
    Bx, By = Bupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # TFSF
    By_phy[sourceidx-1] = By_phy[sourceidx-1] + 0.5*source
    #Bx_phy[sourceidx-1] = Bx_phy[sourceidx-1] + 0.5*source
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # update A
    axion = Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy)
    
    if t == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    # plot
    if t % cycle == 0:
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        im.set_ydata(Ex_phy)
        im2.set_ydata(axion)
        ax.set_title("frame time {}".format(t))
        plt.show()
        plt.pause(0.05)
print('done')

## Plane wave ($B_{external}$, ABC, TFSF)

In [19]:
"""For resonant convrsion, use a plane wave that is close to the deBroglie wavelength 
of axion at around m = 1ueV."""

# Energies of photon ~ 3keV from the Sun
hbar= 6.582e-16 # eVs
freq = 1e10/hbar # ~ 4.6e18 Hz
wavelength = constants.c/freq # 6.58e-11m
# for 20 grids in one wave:
ddx = wavelength/80 # 3.3e-12 m
ddx_natural = ddx/(hbar*constants.c) # 1.67e-5 eV^-1

# since dx*0.5 > dt
dt = ddx_natural*0.5 # 8.35e-6 eV^-1
dt2 = dt**2 # 6.97e-11 eV^-1
dt_si = dt*hbar # 5.5e-21

angFreq = 2*math.pi*freq # ~2.8e19 rad/s
w = angFreq*dt_si # 0.157 rad per grid
nsteps = 2000

# intensity of light (# of photons)
# Erms ~ mu_0 * c* S_avg where S is the poynting vector

Savg = 1.4
Erms = constants.mu_0 * constants.c * Savg # ~527 (SI)
elemC = 5.2909e-19
J2eV = 6.242e18
E0_natural = Erms * elemC * constants.c**2 *  J2eV*hbar # 103041 eV^2

scale2 = 200000 # eV^2
scale = scale2**0.5
E0 = E0_natural/scale2 # say 3000 eV^2 = 0.5 grid units
def get_source(t):
    t0 = 0
    source = E0*np.sin(w*t)
    return source

plt.clf()
plt.plot(np.arange(0,nsteps), get_source(np.arange(0,nsteps)), label='source')
plt.ylim(-2,2)
plt.legend()
plt.show()

In [31]:
"""
For stability reason, I impose two criteria:
1) dt/dx = 0.5
2) dt2 = 0.25
Therefore dx = sqrt(dt2)/0.25 = 1; dt = 0.5
"""

#kappa = 1
@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Bx, By, axion):
    Ex_phy = (Ex + kappa*axion*Bx)/(1+kappa**2*axion**2)
    Ey_phy = (Ey + kappa*axion*By)/(1+kappa**2*axion**2)
    Bx_phy = (Bx - kappa*axion*Ex)/(1+kappa**2*axion**2)
    By_phy = (By - kappa*axion*Ey)/(1+kappa**2*axion**2)
    
    return Ex_phy, Ey_phy, Bx_phy, By_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion):
    Ex = Ex_phy - kappa*axion*Bx_phy
    Ey = Ey_phy - kappa*axion*By_phy
    Bx = Bx_phy + kappa*axion*Ex_phy
    By = By_phy + kappa*axion*Ey_phy
    
    return Ex, Ey, Bx, By

@numba.jit(nopython=True)
def Eupdate1d(Ex, Ey, Bx, By):
    for k in range(1,kmax-1):
        Ex[k] = Ex[k] + 0.5 * (By[k-1] - By[k])
        Ey[k] = Ey[k] + 0.5 * (Bx[k] - Bx[k-1])
    return Ex, Ey
@numba.jit(nopython=True)
def Bupdate1d(Ex, Ey, Bx, By):
    for k in range(0,kmax-1):
        Bx[k] = Bx[k] + 0.5 * (Ey[k+1] - Ey[k])
        By[k] = By[k] + 0.5 * (Ex[k] - Ex[k+1])
    return Bx, By
@numba.jit(nopython=True)
def Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy):
    for k in range(1,kmax-1):
        axion[k] = 2*axion[k] - axion_past[k] + 0.25*(axion[k+1] - 2*axion[k] + axion[k-1])\
        - dt2*kappa*(Ex_phy[k]*Bx_phy[k] + Ey_phy[k]*By_phy[k]) - dt2*(m**2*axion[k])
        
    return axion
    

In [21]:
"""
Everyting here is in natural units, as time is in grid units, dt should also be in grid units
"""

"""
For stability reason, I impose two criteria:
1) dt/dx = 0.5
2) dt2 = 0.25
Therefore dx = sqrt(dt2)/0.25 = 1; dt = 0.5
"""

#kappa = 1
@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Bx, By, axion):
    Ex_phy = (Ex + kappa*axion*Bx)/(1+kappa**2*axion**2)
    Ey_phy = (Ey + kappa*axion*By)/(1+kappa**2*axion**2)
    Bx_phy = (Bx - kappa*axion*Ex)/(1+kappa**2*axion**2)
    By_phy = (By - kappa*axion*Ey)/(1+kappa**2*axion**2)
    
    return Ex_phy, Ey_phy, Bx_phy, By_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion):
    Ex = Ex_phy - kappa*axion*Bx_phy
    Ey = Ey_phy - kappa*axion*By_phy
    Bx = Bx_phy + kappa*axion*Ex_phy
    By = By_phy + kappa*axion*Ey_phy
    
    return Ex, Ey, Bx, By

@numba.jit(nopython=True)
def Eupdate1d(Ex, Ey, Bx, By):
    for k in range(1,kmax-1):
        Ex[k] = Ex[k] + 0.5 * (By[k-1] - By[k])
        Ey[k] = Ey[k] + 0.5 * (Bx[k] - Bx[k-1])
    return Ex, Ey
@numba.jit(nopython=True)
def Bupdate1d(Ex, Ey, Bx, By):
    for k in range(0,kmax-1):
        Bx[k] = Bx[k] + 0.5 * (Ey[k+1] - Ey[k])
        By[k] = By[k] + 0.5 * (Ex[k] - Ex[k+1])
    return Bx, By
@numba.jit(nopython=True)
def Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy):
    for k in range(1,kmax-1):
        axion[k] = 2*axion[k] - axion_past[k] + 0.25*(axion[k+1] - 2*axion[k] + axion[k-1])\
        - dt2*kappa*(Ex_phy[k]*Bx_phy[k] + Ey_phy[k]*By_phy[k]) - dt2*(m**2*axion[k])
        
    return axion


nsteps = 6000
kmax = 4000

# initial values
Ex_phy = Ey_phy = np.zeros(kmax, float)
By_phy = np.zeros(kmax, float)
Bx_phy = np.zeros(kmax, float) # Bext source
axion = np.zeros(kmax, float)
Ex= Ey= Bx= By = np.zeros(kmax, float)

xrange = np.linspace(0,kmax, kmax)

# E source frequency (Energy)
# Energies of photon ~ 3keV from the Sun
hbar= 6.582e-16 # eVs
freq = 3e3/hbar # ~ 4.6e18 Hz
wavelength = constants.c/freq # 6.58e-11m
# for 20 grids in one wave:
ddx = wavelength/400 # ~1e-13 m; sample 400 pts in one wavelength
ddx_natural = ddx/(hbar*constants.c) # ~1e-6 eV^-1

# Stability: since dx*0.5 > dt
dt = ddx_natural*0.5 # 8.35e-6 eV^-1
dt2 = dt**2 # 6.97e-11 eV^-1
dt_si = dt*hbar # 5.5e-21

angFreq = 2*math.pi*freq # ~2.8e19 rad/s
w = angFreq*dt_si # 0.157 rad per grid
nsteps =12000

# E source (lumininosities)
# Erms ~ mu_0 * c* S_avg where S is the poynting vector

Savg = 1400
Erms = (constants.mu_0 * constants.c * Savg)**0.5 # ~700 (SI)
elemC = 5.2909e-19
J2eV = 6.242e18
E0_natural = Erms * elemC * constants.c**2 *  J2eV*hbar # 103041 eV^2

scale2 = 200000 # eV^2
scale = scale2**0.5
E0 = E0_natural/scale2 # say 3000 eV^2 = 0.5 grid units



# # source
# # solar photons have energies around 3keV (2109.07376)
# freq = 3e3/hbar
# wavelength
# wavelength = 3e-4 # 10m # ref: 1542538
# freq = constants.c/wavelength # 3e7Hz
# angFreq = 2*math.pi*freq # ~2e8 rad/s
# w = angFreq*dt_si # rad per grid

# E0_natural = 3000 # eV
# scale2 = 6000 # eV^2
# scale = scale2**0.5
# E0 = 3e3/scale2 # 3keV photon energies from 2109.07376
@numba.jit(nopython=True)
def get_source(t):
    source = E0*np.sin(w*t)
    return source


sourceidx = int(kmax/6)
Bsourceidx = 0
Bext_si = 9 #Tesla
Bext_natural = Bext_si * elemC * constants.c**2 * J2eV * hbar # eV^2
Bext = Bext_natural/scale2

# axion
m = 1/scale # scaled m = 1 eV 
kappa = 1


plt.clf()
plt.close()
cycle = 100
lw=2
fig = plt.figure(figsize=(8,6))
ax = fig.add_axes([.18, .18, .7, .7])
[im] = ax.plot(xrange,Ex_phy,linewidth=lw)
[im2] = ax.plot(xrange,axion,linewidth=lw)
im.set_color('blue')
im2.set_color('orange')
plt.ylim(-1e-12, 1e-12)
#plt.ylim(-1, 1)
plt.legend(['$E_{phy,x}$','axion'])
ax.set_ylabel(r'$\theta$')
init(kmax)

for t in range(0,nsteps+1):
    source = get_source(t)
    source2 = get_source(t+0.5)
    
    if t == 0:
        axion_past = np.zeros(kmax, float)
    # ABC
    Exleft_0 = Ex[1]
    Eyleft_0 = Ey[1]
    Exright_0 = Ex[-2]
    Eyright_0 = Ey[-2]
        
    # update E
    Ex, Ey = Eupdate1d(Ex, Ey, Bx, By)
    # convert to physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    # inject E source
    Ex_phy[sourceidx] = Ex_phy[sourceidx] + 0.5*source2

    #convert back to hat's fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    # ABC
    if t == 0:
        Exleft_ = None
        Eyleft_ = None
        Exright_ = None
        Eyright_ = None
    if t != 0:
        Ex[0] = Exleft_
        Ey[0] = Eyleft_
        Ex[-1] = Exright_
        Ey[-1] = Eyright_
    
    Exleft_ = Exleft_0
    Eyleft_ = Eyleft_0
    Exright_ = Exright_0
    Eyright_ = Eyright_0
    
    # update B
    Bx, By = Bupdate1d(Ex, Ey, Bx, By)
    
    # update physical fields
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    
    # TFSF
    By_phy[sourceidx-1] = By_phy[sourceidx-1] + 0.5*source
    #Bx_phy[sourceidx-1] = Bx_phy[sourceidx-1] + 0.5*source
    
    # update hat fields
    Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)
    
    if t == 2000:
        
        # convert to physical fields
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        # inject B source
        Bx_phy[Bsourceidx:] = Bx_phy[Bsourceidx:] + Bext*np.ones_like(Bx_phy[Bsourceidx:])
        #convert back to hat's fields
        Ex, Ey, Bx, By = phy2hat(Ex_phy, Ey_phy, Bx_phy, By_phy, axion)

        
    # update A
    Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
    axion = Aupdate1d(Ex, Ey, Bx, By, axion, axion_past, Ex_phy, Ey_phy, Bx_phy, By_phy)
    
    if t == 0:
        axion_current = np.zeros(kmax, float)
    axion_past = axion_current
    axion_current = axion
    
    # plot
    if t % cycle == 0:
        Ex_phy, Ey_phy, Bx_phy, By_phy = hat2phy(Ex, Ey, Bx, By, axion)
        #im.set_ydata(Ex_phy) # blue
        im2.set_ydata(axion) # orange
        ax.set_title("frame time {}".format(t))
        plt.savefig('/Users/szechingaudreyfung/Desktop/PHYS 879 HPC/Projects/plots/1dplane_axion/axion{}.png'.format(t))
        plt.show()
        plt.pause(0.05)
print('done')

done


In [12]:
max(axion)

1.921194190669e-13